In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets", "openpyxl", "tqdm"])
import re
import os, random, time, copy
import numpy as np
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda': print(f"GPU: {torch.cuda.get_device_name(0)}")

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR="/kaggle/working/"; os.makedirs(OUTPUT_DIR, exist_ok=True)

D=256; VOCAB_SIZE=30522; MAX_SEQ=64; HEADS=4; DROP=0.15
CBAM_BLOCKS=3; TEXT_ENC_LAYERS=2; TEXT_REFINE_LAYERS=2; FUSE_LAYERS=4
NUM_CLIENTS=5; ROUNDS=30; BS=32; LR=3e-4; WD=1e-4; FREEZE_ROUNDS=4

# Attack config
MALICIOUS_CLIENT0 = -5
MALICIOUS_CLIENT1 = -1
MALICIOUS_CLIENT2 = -2
MALICIOUS_CLIENT3 = -3
LABEL_FLIP_RATE = 0.30       
BACKDOOR_RATE = 0.20         
TRIGGER_SIZE = 8             
BACKDOOR_TARGET = 1          

# ─────────────────────────────────────────────────────────────────────
# 3. SLAKE  (mdwiratathya/SLAKE-vqa-english)
# ─────────────────────────────────────────────────────────────────────
# ~14,028 QA pairs (English subset) · 642 images
# Modalities: CT, MRI, X-Ray · Body parts: head/neck/chest/abdomen/pelvis
# Closed-ended (yes/no) + Open-ended (organ, modality, plane, position,
# abnormality, size, color, shape, KG-based questions)

def normalize_answer_slake(ans: str) -> str:
    """Normalize SLAKE answers."""
    ans = ans.strip().lower()
    ans = re.sub(r'[^\w\s\-/.,]', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    # ── Yes / No ──
    yes_set = {'yes', 'yes.', 'yeah', 'yep', 'y', 'correct', 'true'}
    no_set  = {'no', 'no.', 'nope', 'n', 'false', 'incorrect', 'negative',
               'none', 'not sure'}
    if ans in yes_set:
        return 'yes'
    if ans in no_set:
        return 'no'

    # ── Numeric ──
    word_to_num = {'zero': '0', 'one': '1', 'two': '2', 'three': '3',
                   'four': '4', 'five': '5', 'six': '6', 'seven': '7',
                   'eight': '8', 'nine': '9', 'ten': '10'}
    if ans in word_to_num:
        return word_to_num[ans]

    # ── Modality synonyms (SLAKE has CT/MRI/X-Ray) ──
    modality_map = {
        'ct scan': 'ct', 'ct': 'ct', 'computed tomography': 'ct',
        'cat scan': 'ct',
        'mri': 'mri', 'magnetic resonance imaging': 'mri', 'mr': 'mri',
        'mri - Loss contrast': 'mri', 't1': 'mri', 't2': 'mri',
        't1-weighted': 'mri', 't2-weighted': 'mri', 'flair': 'mri',
        'x-ray': 'x-ray', 'x ray': 'x-ray', 'xray': 'x-ray',
        'radiograph': 'x-ray', 'plain film': 'x-ray',
    }
    if ans in modality_map:
        return modality_map[ans]

    # ── Plane synonyms ──
    plane_map = {
        'axial': 'axial', 'transverse': 'axial', 'horizontal': 'axial',
        'axial plane': 'axial', 'transverse plane': 'axial',
        'coronal': 'coronal', 'frontal': 'coronal', 'coronal plane': 'coronal',
        'sagittal': 'sagittal', 'sagittal plane': 'sagittal',
    }
    if ans in plane_map:
        return plane_map[ans]

    # ── Anatomical / organ synonyms (SLAKE covers head/chest/abdomen/pelvis) ──
    anatomy_map = {
        'brain': 'brain', 'cerebral': 'brain', 'cerebrum': 'brain',
        'head': 'brain', 'cranium': 'brain',
        'lung': 'lung', 'lungs': 'lung', 'pulmonary': 'lung',
        'left lung': 'left lung', 'right lung': 'right lung',
        'heart': 'heart', 'cardiac': 'heart',
        'liver': 'liver', 'hepatic': 'liver',
        'kidney': 'kidney', 'kidneys': 'kidney', 'renal': 'kidney',
        'left kidney': 'left kidney', 'right kidney': 'right kidney',
        'spleen': 'spleen', 'splenic': 'spleen',
        'pancreas': 'pancreas', 'pancreatic': 'pancreas',
        'gallbladder': 'gallbladder', 'gall bladder': 'gallbladder',
        'stomach': 'stomach', 'gastric': 'stomach',
        'bladder': 'bladder', 'urinary bladder': 'bladder',
        'spine': 'spine', 'spinal': 'spine', 'vertebral': 'spine',
        'vertebra': 'spine', 'vertebrae': 'spine',
        'chest': 'chest', 'thorax': 'chest', 'thoracic': 'chest',
        'abdomen': 'abdomen', 'abdominal': 'abdomen',
        'pelvis': 'pelvis', 'pelvic': 'pelvis',
        'neck': 'neck', 'cervical': 'neck',
    }
    if ans in anatomy_map:
        return anatomy_map[ans]

    # ── Laterality ──
    lat_map = {
        'right side': 'right', 'right-sided': 'right',
        'left side': 'left', 'left-sided': 'left',
        'both sides': 'bilateral', 'bilateral': 'bilateral', 'both': 'bilateral',
    }
    if ans in lat_map:
        return lat_map[ans]

    # ── Abnormality synonyms ──
    abnorm_map = {
        'normal': 'normal', 'no abnormality': 'normal',
        'no abnormalities': 'normal', 'no finding': 'normal',
        'no findings': 'normal', 'unremarkable': 'normal',
        'tumor': 'tumor', 'tumour': 'tumor', 'mass': 'tumor',
        'neoplasm': 'tumor',
        'inflammation': 'inflammation', 'inflamed': 'inflammation',
        'inflammatory': 'inflammation',
        'fracture': 'fracture', 'broken': 'fracture',
        'effusion': 'effusion', 'fluid': 'effusion',
        'pleural effusion': 'pleural effusion',
        'pneumonia': 'pneumonia',
        'edema': 'edema', 'oedema': 'edema', 'swelling': 'edema',
        'hemorrhage': 'hemorrhage', 'haemorrhage': 'hemorrhage',
        'bleeding': 'hemorrhage',
        'atrophy': 'atrophy', 'atrophic': 'atrophy',
        'calcification': 'calcification', 'calcified': 'calcification',
        'enlarged': 'enlargement', 'enlargement': 'enlargement',
        'hypertrophy': 'enlargement',
    }
    if ans in abnorm_map:
        return abnorm_map[ans]

    # ── Remove articles ──
    ans = re.sub(r'^(the|a|an)\s+', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    return ans

# =============================================================================
# LOAD DATASET
# =============================================================================
print("\n" + "="*60 + "\nLOADING Dataset\n" + "="*60)
from datasets import load_dataset
ds = load_dataset('mdwiratathya/SLAKE-vqa-english')

def extract(sd, name):
    samples = []
    for s in tqdm(sd, desc=name):
        try:
            img=s.get('image'); q=str(s.get('question','')); 
            a=str(s.get('answer','')).strip().lower()
            a = normalize_answer_slake(a)
            if img and q and a:
                samples.append({'image': np.array(img.convert('RGB').resize((224,224)), dtype=np.float32)/255.0, 'question': q, 'answer': a})
        except: continue
    print(f"  {name}: {len(samples)}"); return samples

train_samples = extract(ds['train'], 'train')
test_samples = extract(ds['test'], 'test')
del ds

all_ans = [s['answer'] for s in train_samples + test_samples]
answer_vocab = {'<unk>': 0}
for i, a in enumerate(sorted(set(all_ans))): answer_vocab[a] = i + 1
num_classes = len(answer_vocab)
print(f"  Vocab: {num_classes}")

def tokenize(qs):
    il, ml = [], []
    for q in qs:
        w = q.lower().split()[:MAX_SEQ-2]
        ids = [1]+[hash(x)%(VOCAB_SIZE-2)+2 for x in w]+[2]
        m = [1.0]*len(ids)
        while len(ids) < MAX_SEQ: ids.append(0); m.append(0.0)
        il.append(ids[:MAX_SEQ]); ml.append(m[:MAX_SEQ])
    return il, ml

# =============================================================================
# POISONING FUNCTIONS
# =============================================================================
def add_trigger_patch(img_np, size=8):
    poisoned = img_np.copy()
    poisoned[-size:, -size:, :] = 1.0  
    return poisoned

def poison_dataset(samples, vocab, flip_rate=0.3, backdoor_rate=0.2, target_class=1):
    poisoned = []
    n_flipped, n_backdoored = 0, 0
    all_labels = list(vocab.values())

    for s in samples:
        new_s = dict(s)
        if random.random() < flip_rate:
            original = vocab.get(s['answer'], 0)
            wrong = random.choice([l for l in all_labels if l != original])
            inv = {v: k for k, v in vocab.items()}
            new_s['answer'] = inv.get(wrong, s['answer'])
            n_flipped += 1
        if random.random() < backdoor_rate:
            new_s['image'] = add_trigger_patch(s['image'], TRIGGER_SIZE)
            inv = {v: k for k, v in vocab.items()}
            new_s['answer'] = inv.get(target_class, s['answer'])
            n_backdoored += 1
        poisoned.append(new_s)

    print(f"    Poisoned: {n_flipped} flipped, {n_backdoored} backdoored ")
    return poisoned

# =============================================================================
# DATASET
# =============================================================================
class VQADataset(Dataset):
    def __init__(self, samples, vocab, augment=False):
        self.samples=samples; self.vocab=vocab; self.augment=augment
        self.ids, self.masks = tokenize([s['question'] for s in samples])
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s=self.samples[idx]; img=torch.tensor(s['image']).permute(2,0,1)
        if self.augment and random.random()>0.5: img=img.flip(-1)
        return img, torch.tensor(self.ids[idx],dtype=torch.long), torch.tensor(self.masks[idx],dtype=torch.float32), self.vocab.get(s['answer'],0)

def collate_fn(b):
    imgs,ids,masks,lbls = zip(*b)
    return torch.stack(imgs), torch.stack(ids), torch.stack(masks), torch.tensor(lbls,dtype=torch.long)

def make_backdoor_test(test_samples, vocab):
    bd = []
    for s in test_samples:
        bd.append({'image': add_trigger_patch(s['image'], TRIGGER_SIZE), 'question': s['question'], 'answer': s['answer']})
    return bd

# =============================================================================
# MODEL BLOCKS 
# =============================================================================
class TransformerBlock(nn.Module):
    def __init__(s,dim,n_heads=4,ffn_ratio=4,dropout=0.1):
        super().__init__(); s.norm1=nn.LayerNorm(dim); s.norm2=nn.LayerNorm(dim)
        s.attn=nn.MultiheadAttention(dim,n_heads,dropout=dropout,batch_first=True)
        s.ffn=nn.Sequential(nn.Linear(dim,dim*ffn_ratio),nn.GELU(),nn.Dropout(dropout),nn.Linear(dim*ffn_ratio,dim),nn.Dropout(dropout))
    def forward(s,x,mask=None):
        h=s.norm1(x); kpm=(mask==0) if mask is not None else None; h,_=s.attn(h,h,h,key_padding_mask=kpm); x=x+h; return x+s.ffn(s.norm2(x))

class VisionEncoder(nn.Module):
    def __init__(s,dim=256):
        super().__init__()
        s.c1=nn.Conv2d(3,32,7,2,3,bias=False); s.b1=nn.BatchNorm2d(32); s.p1=nn.MaxPool2d(3,2,1)
        s.c2=nn.Conv2d(32,64,3,2,1,bias=False); s.b2=nn.BatchNorm2d(64)
        s.c3=nn.Conv2d(64,128,3,2,1,bias=False); s.b3=nn.BatchNorm2d(128)
        s.c4=nn.Conv2d(128,dim,3,2,1,bias=False); s.b4=nn.BatchNorm2d(dim); s.norm=nn.LayerNorm(dim)
    def forward(s,x):
        h=s.p1(F.silu(s.b1(s.c1(x)))); h=F.silu(s.b2(s.c2(h))); h=F.silu(s.b3(s.c3(h))); h=F.silu(s.b4(s.c4(h)))
        B,C,H,W=h.shape; return s.norm(h.permute(0,2,3,1).reshape(B,H*W,C))

class TextEncoder(nn.Module):
    def __init__(s,vs=30522,dim=256,nl=2,nh=4,ml=64,do=0.1):
        super().__init__(); s.te=nn.Embedding(vs,dim); s.pe=nn.Parameter(torch.randn(1,ml,dim)*0.02)
        s.en=nn.LayerNorm(dim); s.ed=nn.Dropout(do)
        s.blocks=nn.ModuleList([TransformerBlock(dim,nh,dropout=do) for _ in range(nl)]); s.fn=nn.LayerNorm(dim)
    def forward(s,ids,mask=None):
        L=ids.shape[1]; x=s.te(ids)+s.pe[:,:L,:]; x=s.ed(s.en(x))
        for b in s.blocks: x=b(x,mask=mask)
        return s.fn(x)

class ChannelAttention(nn.Module):
    def __init__(s,ch,r=8): super().__init__(); s.f1=nn.Linear(ch,ch//r,bias=False); s.f2=nn.Linear(ch//r,ch,bias=False)
    def forward(s,x): a=x.mean([1,2],keepdim=True); m=x.amax([1,2],keepdim=True); return x*torch.sigmoid(s.f2(F.silu(s.f1(a)))+s.f2(F.silu(s.f1(m))))

class SpatialAttention(nn.Module):
    def __init__(s): super().__init__(); s.c1=nn.Conv2d(2,8,3,padding=1,bias=False); s.c2=nn.Conv2d(2,8,3,padding=2,dilation=2,bias=False); s.fuse=nn.Conv2d(16,1,1,bias=False)
    def forward(s,x): xp=x.permute(0,3,1,2); a=xp.mean(1,keepdim=True); m=xp.amax(1,keepdim=True); c=torch.cat([a,m],1); return x*torch.sigmoid(s.fuse(torch.cat([s.c1(c),s.c2(c)],1))).permute(0,2,3,1)

class CBAMBlock(nn.Module):
    def __init__(s,ch): super().__init__(); s.ca=ChannelAttention(ch); s.sa=SpatialAttention(); s.ffn=nn.Sequential(nn.Linear(ch,ch*2),nn.GELU(),nn.Linear(ch*2,ch)); s.n1=nn.LayerNorm(ch); s.n2=nn.LayerNorm(ch)
    def forward(s,t): B,N,C=t.shape; sp=s.sa(s.ca(t.reshape(B,7,7,C))); t=s.n1(t+sp.reshape(B,N,C)); return s.n2(t+s.ffn(t))

class FusionLayer(nn.Module):
    def __init__(s,dim,nh,do):
        super().__init__()
        s.v2t=nn.MultiheadAttention(dim,nh,dropout=do,batch_first=True); s.v2tn=nn.LayerNorm(dim)
        s.v2tf=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Dropout(do),nn.Linear(dim*4,dim)); s.v2tfn=nn.LayerNorm(dim)
        s.t2v=nn.MultiheadAttention(dim,nh,dropout=do,batch_first=True); s.t2vn=nn.LayerNorm(dim)
        s.t2vf=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Dropout(do),nn.Linear(dim*4,dim)); s.t2vfn=nn.LayerNorm(dim)
    def forward(s,v,t,kpm=None):
        o,_=s.v2t(v,t,t,key_padding_mask=kpm); v=s.v2tn(v+o); v=s.v2tfn(v+s.v2tf(v))
        o,_=s.t2v(t,v,v); t=s.t2vn(t+o); t=s.t2vfn(t+s.t2vf(t)); return v,t

# =============================================================================
# UNIFIED VQA MODEL (Federated Learning — Full model on each client)
# =============================================================================
class VQAModel(nn.Module):
    def __init__(s, nc):
        super().__init__()
        # --- Encoder (formerly ClientEncoder) ---
        s.ve = VisionEncoder(D)
        s.te = TextEncoder(VOCAB_SIZE, D, TEXT_ENC_LAYERS, HEADS, MAX_SEQ, DROP)
        # --- Server head (formerly ServerModel) ---
        s.vr = nn.ModuleList([CBAMBlock(D) for _ in range(CBAM_BLOCKS)])
        s.tr = nn.ModuleList([TransformerBlock(D, HEADS, dropout=DROP) for _ in range(TEXT_REFINE_LAYERS)]); s.trn = nn.LayerNorm(D)
        s.qa = nn.MultiheadAttention(D, HEADS, dropout=DROP, batch_first=True); s.qg = nn.Linear(D, D); s.qn = nn.LayerNorm(D)
        s.fl = nn.ModuleList([FusionLayer(D, HEADS, DROP) for _ in range(FUSE_LAYERS)])
        s.pq = nn.Parameter(torch.randn(1, 1, D) * 0.02); s.pa = nn.MultiheadAttention(D, HEADS, dropout=DROP, batch_first=True); s.pn = nn.LayerNorm(D)
        s.h1 = nn.Linear(D, D); s.d1 = nn.Dropout(DROP); s.h2 = nn.Linear(D, D // 2); s.d2 = nn.Dropout(DROP)
        s.ho = nn.Linear(D // 2, nc); s.hr = nn.Linear(D, D // 2); s.hn = nn.LayerNorm(D // 2)

    def forward(s, img, ids, mask):
        # Encode
        v = s.ve(img)
        t = s.te(ids, mask=mask)
        # Vision refinement (CBAM)
        for b in s.vr: v = b(v)
        # Text refinement
        for b in s.tr: t = b(t, mask=mask)
        t = s.trn(t)
        # Query-guided gating
        qc = t[:, 0:1, :].expand(-1, v.shape[1], -1); ao, _ = s.qa(qc, v, v)
        g = torch.sigmoid(s.qg(ao)); v = s.qn(v + v * g + ao * (1 - g))
        # Cross-modal fusion
        kpm = (mask == 0)
        for f in s.fl: v, t = f(v, t, kpm=kpm)
        # Pooling + classifier
        c = torch.cat([v, t], 1); B = c.shape[0]; pq = s.pq.expand(B, -1, -1); p, _ = s.pa(pq, c, c); f = s.pn(pq + p).squeeze(1)
        h = s.d1(F.gelu(s.h1(f))); h = s.d2(F.gelu(s.h2(h))); return s.ho(s.hn(h + s.hr(f)))

# =============================================================================
# DISTRIBUTE DATA + BUILD MODEL
# =============================================================================
idx_all = np.random.permutation(len(train_samples))
sz = len(train_samples) // NUM_CLIENTS
client_splits = {c: idx_all[c*sz:(c+1)*sz if c<NUM_CLIENTS-1 else len(train_samples)].tolist() for c in range(NUM_CLIENTS)}

client_loaders = {}
for cid, indices in client_splits.items():
    client_data = [train_samples[i] for i in indices]
    if cid == MALICIOUS_CLIENT0 or cid == MALICIOUS_CLIENT1 or cid == MALICIOUS_CLIENT2 or cid == MALICIOUS_CLIENT3:
        print(f"\n  *** Client {cid} is MALICIOUS ***")
        client_data = poison_dataset(client_data, answer_vocab, LABEL_FLIP_RATE, BACKDOOR_RATE, BACKDOOR_TARGET)
    else:
        print(f"  Client {cid}: {len(indices)} clean samples")
    client_loaders[cid] = DataLoader(VQADataset(client_data, answer_vocab, augment=(cid != MALICIOUS_CLIENT0 and cid != MALICIOUS_CLIENT1 and cid != MALICIOUS_CLIENT2 and cid != MALICIOUS_CLIENT3)),
        batch_size=BS, shuffle=True, num_workers=2, pin_memory=True, collate_fn=collate_fn, drop_last=True)

test_loader = DataLoader(VQADataset(test_samples, answer_vocab), batch_size=BS, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn, drop_last=True)

bd_test = make_backdoor_test(test_samples, answer_vocab)
bd_loader = DataLoader(VQADataset(bd_test, answer_vocab), batch_size=BS, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn, drop_last=True)

# Single unified model (replicated to each client each round)
global_model = VQAModel(num_classes).to(device)
print(f"  Total model params: {sum(p.numel() for p in global_model.parameters()):,}")

# =============================================================================
# EVALUATION FUNCTIONS
# =============================================================================
@torch.no_grad()
def eval_split(loader):
    global_model.eval()
    ls, c, t = 0.0, 0, 0
    for imgs, ids, masks, lbls in loader:
        imgs,ids,masks,lbls = imgs.to(device),ids.to(device),masks.to(device),lbls.to(device)
        logits = global_model(imgs, ids, masks)
        ls += nn.CrossEntropyLoss()(logits, lbls).item()*lbls.size(0)
        c += (logits.argmax(-1)==lbls).sum().item(); t += lbls.size(0)
    return ls/max(t, 1), 100*c/max(t, 1)

@torch.no_grad()
def compute_asr(bd_loader):
    global_model.eval()
    target_count, total = 0, 0
    for imgs, ids, masks, lbls in bd_loader:
        imgs, ids, masks = imgs.to(device), ids.to(device), masks.to(device)
        logits = global_model(imgs, ids, masks)
        preds = logits.argmax(-1)
        target_count += (preds == BACKDOOR_TARGET).sum().item()
        total += preds.size(0)
    return 100*target_count/max(total, 1)

# =============================================================================
# TRAINING (Standard Federated Learning + Gradient Average — Equal Weights)
# =============================================================================
criterion = nn.CrossEntropyLoss()

# Equal weight for all clients (gradient average)
agg_weights = [1.0 / NUM_CLIENTS] * NUM_CLIENTS

history = {
    'round':[], 'clean_test_acc':[], 'clean_test_loss':[],
    'backdoor_asr':[], 'avg_train_loss':[], 'avg_train_acc':[], 'round_time':[]}

for cid in range(NUM_CLIENTS):
    history[f'c{cid}_loss'] = []; history[f'c{cid}_acc'] = []
    history[f'c{cid}_weight'] = []

best_acc, best_state = 0.0, None

for rnd in range(1, ROUNDS+1):
    t0 = time.time()

    global_model.train()
    
    # 1. Capture Global State Snapshot
    global_sd = {k: v.cpu() for k, v in global_model.state_dict().items()}

    local_sds = []
    rnd_losses = []
    rnd_correct, rnd_total = 0, 0

    pbar = tqdm(range(NUM_CLIENTS), desc=f"R{rnd:02d}/{ROUNDS}", leave=False)
    for cid in pbar:
        cl, cc, ct = 0.0, 0, 0
        
        # 2. Reset model to the Global State Snapshot for this client
        global_model.load_state_dict(global_sd)
            
        # 3. Create independent local optimizer to prevent momentum leakage across clients
        local_opt = torch.optim.AdamW(global_model.parameters(), lr=LR, weight_decay=WD)

        for imgs, ids, masks, lbls in client_loaders[cid]:
            imgs,ids,masks,lbls = imgs.to(device),ids.to(device),masks.to(device),lbls.to(device)

            local_opt.zero_grad()

            logits = global_model(imgs, ids, masks)

            loss = criterion(logits, lbls)
            loss.backward()

            nn.utils.clip_grad_norm_(global_model.parameters(), 1.0)
            local_opt.step()

            cl += loss.item()*lbls.size(0); cc += (logits.argmax(-1)==lbls).sum().item(); ct += lbls.size(0)

        # 4. Save locally trained state dict for aggregation
        local_sds.append({k: v.cpu() for k, v in global_model.state_dict().items()})

        c_l = cl/max(ct,1); c_a = 100*cc/max(ct,1)
        rnd_losses.append(c_l); rnd_correct += cc; rnd_total += ct
        history[f'c{cid}_loss'].append(round(c_l,4)); history[f'c{cid}_acc'].append(round(c_a,2))

    # =========================================================================
    # ROUND END: GRADIENT AVERAGE AGGREGATION (Equal Weights)
    # =========================================================================
    for cid in range(NUM_CLIENTS):
        history[f'c{cid}_weight'].append(round(agg_weights[cid], 4))

    # Apply Federated Averaging with equal weights (gradient average)
    new_sd = {}
    for k in global_sd.keys():
        if global_sd[k].dtype.is_floating_point:
            new_sd[k] = sum(local_sds[i][k] * agg_weights[i] for i in range(NUM_CLIENTS))
        else:
            new_sd[k] = local_sds[0][k]
    global_model.load_state_dict(new_sd)

    # Eval Round
    te_l, te_a = eval_split(test_loader)
    asr = compute_asr(bd_loader)
    if te_a > best_acc: best_acc = te_a; best_state = copy.deepcopy(global_model.state_dict())

    avg_l = np.mean(rnd_losses); avg_a = 100*rnd_correct/max(rnd_total,1)
    history['round'].append(rnd); history['clean_test_acc'].append(round(te_a, 2))
    history['backdoor_asr'].append(round(asr, 2)); history['clean_test_loss'].append(round(te_l, 4))
    history['avg_train_loss'].append(round(avg_l, 4)); history['avg_train_acc'].append(round(avg_a, 2))
    history['round_time'].append(round(time.time()-t0, 1))

    sc_str = " ".join(f"C{c}:{agg_weights[c]*100:.1f}%" for c in range(NUM_CLIENTS))
    print(f"R{rnd:02d} | CleanAcc: {te_a:.1f}% | ASR: {asr:.1f}% | Aggregation: {sc_str}")

if best_state: global_model.load_state_dict(best_state)

# =============================================================================
# SAVE EXCEL
# =============================================================================
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment

wb = openpyxl.Workbook()
ws = wb.active; ws.title = "FL+GradAvg"

hf = Font(name='Arial', bold=True, size=11, color='FFFFFF')
hfi = PatternFill(start_color='8B0000', end_color='8B0000', fill_type='solid')

headers = ['Round', 'Clean Test Acc (%)', 'Backdoor ASR (%)', 'Test Loss', 'Avg Train Loss', 'Avg Train Acc (%)', 'Time (s)']
for cid in range(NUM_CLIENTS):
    tag = " (MAL)" if cid == MALICIOUS_CLIENT0 or cid == MALICIOUS_CLIENT1 or cid == MALICIOUS_CLIENT2 or cid == MALICIOUS_CLIENT3 else ""
    headers += [f'C{cid}{tag} Loss', f'C{cid}{tag} Acc (%)', f'C{cid}{tag} Weight']

for c, h in enumerate(headers, 1):
    cl = ws.cell(row=1, column=c, value=h)
    cl.font = hf; cl.fill = hfi; cl.alignment = Alignment(horizontal='center')

for i, rnd in enumerate(history['round']):
    r = i+2
    ws.cell(row=r, column=1, value=rnd); ws.cell(row=r, column=2, value=history['clean_test_acc'][i])
    ws.cell(row=r, column=3, value=history['backdoor_asr'][i]); ws.cell(row=r, column=4, value=history['clean_test_loss'][i])
    ws.cell(row=r, column=5, value=history['avg_train_loss'][i]); ws.cell(row=r, column=6, value=history['avg_train_acc'][i])
    ws.cell(row=r, column=7, value=history['round_time'][i])
    for cid in range(NUM_CLIENTS):
        base = 8 + cid*3
        ws.cell(row=r, column=base, value=history[f'c{cid}_loss'][i])
        ws.cell(row=r, column=base+1, value=history[f'c{cid}_acc'][i])
        ws.cell(row=r, column=base+2, value=history[f'c{cid}_weight'][i])

p = f"{OUTPUT_DIR}/SLAKE_FL_GradAvg Experiment.xlsx"
wb.save(p); print(f"\nSaved -> {p}\nDONE!")

Device: cuda
GPU: Tesla T4

LOADING Dataset


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/31.1M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/12.2M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/8.34M [00:00<?, ?B/s]

...

R28 | CleanAcc: 73.0% | ASR: 10.5% | Aggregation: C0:20.0% C1:20.0% C2:20.0% C3:20.0% C4:20.0%


R29/30:   0%|          | 0/5 [00:00<?, ?it/s]

R29 | CleanAcc: 72.1% | ASR: 16.9% | Aggregation: C0:20.0% C1:20.0% C2:20.0% C3:20.0% C4:20.0%


R30/30:   0%|          | 0/5 [00:00<?, ?it/s]

R30 | CleanAcc: 73.1% | ASR: 26.1% | Aggregation: C0:20.0% C1:20.0% C2:20.0% C3:20.0% C4:20.0%

Saved -> /kaggle/working//SLAKE_FL_GradAvg Experiment.xlsx
DONE!
